<a href="https://colab.research.google.com/github/shahaanabbas-13/Quantum-Diabetes-Prediction/blob/main/notebooks/04_Feature_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Feature selection and dimensionality reduction
Compare Mutual Information, RFE and PCA. All selectors are fitted only on training folds in a pipeline. This notebook reports CV comparisons for a logistic-regression reference model.

In [1]:
%pip -q install scikit-learn

In [3]:
from pathlib import Path
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif, SelectKBest, RFE
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve)
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
# If running from a notebook in /notebooks, move to project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "dataset" / "diabetes.csv"
if not DATA_PATH.exists():
    # Colab fallback: upload diabetes.csv into the current session or mount Drive.
    DATA_PATH = Path("/content/diabetes.csv")
df = pd.read_csv(DATA_PATH)
TARGET = "Outcome"
FEATURES = [c for c in df.columns if c != TARGET]
INVALID_ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_validate
X=df.drop(columns=TARGET).replace({c:{0:np.nan} for c in INVALID_ZERO_COLS})
y=df[TARGET].astype(int)
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
selector_options={
 "All 8 features": "passthrough",
 "Mutual Information (4)": SelectKBest(score_func=mutual_info_classif,k=4),
 "RFE (4)": RFE(estimator=LogisticRegression(max_iter=2000),n_features_to_select=4),
 "PCA (4)": PCA(n_components=4,random_state=42)
}
rows=[]
for name,selector in selector_options.items():
    pipe=Pipeline([
      ("imputer",SimpleImputer(strategy="median")),
      ("scale",StandardScaler()),
      ("selector",selector),
      ("model",LogisticRegression(max_iter=2000,class_weight="balanced"))
    ])
    s=cross_validate(pipe,X,y,cv=cv,scoring={"accuracy":"accuracy","recall":"recall","f1":"f1","roc_auc":"roc_auc"})
    rows.append({"representation":name,**{m:s[f"test_{m}"].mean() for m in ["accuracy","recall","f1","roc_auc"]}})
feature_results=pd.DataFrame(rows)
display(feature_results)

# Create results directory
RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save feature selection results
RESULTS_PATH = RESULTS_DIR / "feature_selection_cv.csv"

feature_results.to_csv(RESULTS_PATH, index=False)

print("Feature selection results saved successfully!")
print("Location:", RESULTS_PATH)

# Verify saved results
display(pd.read_csv(RESULTS_PATH))


,representation,accuracy,recall,f1,roc_auc
0,All 8 features,0.747313,0.727393,0.668809,0.836225
1,Mutual Information (4),0.740778,0.701328,0.654680,0.829936
2,RFE (4),0.764222,0.723899,0.684152,0.840164
3,PCA (4),0.730396,0.708805,0.648018,0.810744


Feature selection results saved successfully!
Location: /content/results/feature_selection_cv.csv


,representation,accuracy,recall,f1,roc_auc
0,All 8 features,0.747313,0.727393,0.668809,0.836225
1,Mutual Information (4),0.740778,0.701328,0.654680,0.829936
2,RFE (4),0.764222,0.723899,0.684152,0.840164
3,PCA (4),0.730396,0.708805,0.648018,0.810744


For quantum circuits, use a four-dimensional representation. Mutual Information and RFE retain original features; PCA creates transformed components. Fit selectors within each training fold to avoid leakage.